# Pokemon Battle Prediction - Model Training & Evaluation

## Project Overview
This notebook focuses on training and evaluating machine learning models to predict Pokemon battle outcomes using the preprocessed dataset.

## Objective
To develop and compare multiple machine learning models for predicting which Pokemon will win in a battle, achieving high accuracy and robust performance.

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading

In [4]:
preprocessed_df=pd.read_csv('pre-processed-dataset.csv')
preprocessed_df

,Legendary_first,Legendary_second,first_winner,HP_diff,Attack_diff,Defense_diff,Sp_Atk_diff,Sp_Def_diff,Height_diff,Weight_diff,...,Psychic_second,Ice_second,Bug_second,Ghost_second,Steel_second,Dragon_second,Dark_second,Fairy_second,First_pokemon_attacks_first,Second_pokemon_attacks_first
0,0,0,0,0.455556,0.455738,0.533175,0.440141,0.521845,0.484211,0.523408,...,0,0,0,0,0,0,1,0,0,1
1,1,1,0,0.500000,0.347541,0.466825,0.556338,0.592233,0.501754,0.468640,...,0,0,0,0,0,0,0,0,1,1
2,0,0,0,0.455556,0.360656,0.533175,0.334507,0.521845,0.484211,0.483754,...,1,0,0,0,0,0,0,0,1,1
3,0,0,0,0.417778,0.213115,0.390995,0.528169,0.376214,0.466667,0.445468,...,0,0,0,0,0,1,0,0,0,1
4,0,0,1,0.611111,0.639344,0.260664,0.862676,0.109223,0.512281,0.507873,...,0,0,1,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38049,0,0,0,0.511111,0.350820,0.509479,0.500000,0.497573,0.470175,0.490021,...,0,0,0,0,0,0,0,0,1,1
38050,1,0,1,0.655556,0.737705,0.580569,0.774648,0.728155,0.596491,0.669809,...,0,0,0,0,0,0,0,0,1,0
38051,0,0,1,0.555556,0.573770,0.509479,0.440141,0.509709,0.501754,0.504555,...,0,0,0,0,0,0,0,0,1,0
38052,0,0,0,0.471111,0.262295,0.604265,0.580986,0.521845,0.494737,0.493760,...,0,0,0,0,0,0,0,0,0,1


---

## Data Splitting Strategy

We use a three-way split to ensure proper model evaluation:

In [5]:
from sklearn.model_selection import train_test_split

# Step 1: Separate features and target
X = preprocessed_df.drop(columns=['first_winner'])  # Features
y = preprocessed_df['first_winner']                # Target

# Step 2: First split - train+val and test (80% train+val, 20% test)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Step 3: Second split - train and val (from the 80%, split 75% train / 25% val → total: 60% train, 20% val)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Now:
# - X_train, y_train: training set (60%)
# - X_val, y_val: validation set (20%)
# - X_test, y_test: test set (20%)

**Final Split:**
- Training set: 60% (for model training)
- Validation set: 20% (for model selection and hyperparameter tuning)
- Test set: 20% (for final unbiased evaluation)

**Key Features:**
- **Stratified sampling** ensures balanced class distribution across all splits
- **Fixed random state** for reproducible results
- **Separate test set** prevents data leakage during model development

---

## Baseline Model Comparison

We evaluate three different algorithms to understand their performance characteristics:

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize models
log_reg = LogisticRegression()
tree = DecisionTreeClassifier()
rf = RandomForestClassifier()

# Train and evaluate on validation set
for model in [log_reg, tree, rf]:
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    print(f"{model.__class__.__name__} Accuracy: {accuracy_score(y_val, preds):.2f}")
    print(classification_report(y_val, preds))


LogisticRegression Accuracy: 0.94
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      4028
           1       0.93      0.95      0.94      3583

    accuracy                           0.94      7611
   macro avg       0.94      0.94      0.94      7611
weighted avg       0.94      0.94      0.94      7611

DecisionTreeClassifier Accuracy: 0.93
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      4028
           1       0.93      0.92      0.92      3583

    accuracy                           0.93      7611
   macro avg       0.93      0.93      0.93      7611
weighted avg       0.93      0.93      0.93      7611

RandomForestClassifier Accuracy: 0.95
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      4028
           1       0.94      0.96      0.95      3583

    accuracy                           0.95      7611
   macro avg      

**Key Observations:**
- All models achieve excellent performance (>93% accuracy)
- Random Forest shows the best initial performance
- Strong feature engineering leads to consistent results across different algorithms
- Balanced precision and recall indicate robust classification

---

## Hyperparameter Optimization

Since Random Forest performed best, we optimize it using GridSearchCV:

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define a parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# Create the base model
rf = RandomForestClassifier(random_state=42)

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)

# Best model
best_rf = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)


Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


**Hyperparameter Tuning Strategy:**
- **5-fold cross-validation** for robust parameter evaluation
- **Grid search** explores all parameter combinations
- **Accuracy scoring** optimizes for overall prediction correctness
- **Parallel processing** (`n_jobs=-1`) for faster computation

---

## Model Performance Analysis

### Validation Set Performance

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

val_preds = best_rf.predict(X_val)
print(classification_report(y_val, val_preds))
print(confusion_matrix(y_val, val_preds))

              precision    recall  f1-score   support

           0       0.96      0.95      0.95      4028
           1       0.94      0.96      0.95      3583

    accuracy                           0.95      7611
   macro avg       0.95      0.95      0.95      7611
weighted avg       0.95      0.95      0.95      7611

[[3819  209]
 [ 158 3425]]


### Test Set Performance (Final Evaluation)

In [9]:
test_preds = best_rf.predict(X_test)
print("Final Test Set Results:")
print(classification_report(y_test, test_preds))
print(confusion_matrix(y_test, test_preds))


Final Test Set Results:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      4028
           1       0.94      0.95      0.95      3583

    accuracy                           0.95      7611
   macro avg       0.95      0.95      0.95      7611
weighted avg       0.95      0.95      0.95      7611

[[3817  211]
 [ 162 3421]]


In [10]:
import joblib
joblib.dump(best_rf, 'best_random_forest_model.pkl')

['best_random_forest_model.pkl']

---

## Advanced Model: XGBoost

We also evaluate XGBoost, a state-of-the-art gradient boosting algorithm:

In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

xgb_val_preds = xgb_model.predict(X_val)

print("XGBoost Validation Accuracy:", accuracy_score(y_val, xgb_val_preds))
print(classification_report(y_val, xgb_val_preds))

D:\python_envs\myenv\lib\site-packages\xgboost\training.py:183: UserWarning: [20:23:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Validation Accuracy: 0.9558533701221915
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      4028
           1       0.95      0.96      0.95      3583

    accuracy                           0.96      7611
   macro avg       0.96      0.96      0.96      7611
weighted avg       0.96      0.96      0.96      7611



### XGBoost Test Performance

In [16]:
test_preds = xgb_model.predict(X_test)
print("XGBoost Test Accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))

XGBoost Test Accuracy: 0.9545394823282092
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      4028
           1       0.95      0.95      0.95      3583

    accuracy                           0.95      7611
   macro avg       0.95      0.95      0.95      7611
weighted avg       0.95      0.95      0.95      7611



In [17]:
joblib.dump(xgb_model, 'xgb_boost_model.pkl')

['xgb_boost_model.pkl']

---

## Final Model Comparison

| Model | Validation Accuracy | Test Accuracy | Key Strengths |
|-------|-------------------|---------------|---------------|
| **Optimized Random Forest** | 95.0% | 95.0% | Robust, interpretable, handles mixed data types |
| **XGBoost** | **95.6%** | **95.4%** | Slight performance edge, gradient boosting |

## Key Insights

### Model Performance
- **Exceptional accuracy**: Both final models achieve >95% accuracy
- **Consistent performance**: Similar validation and test scores indicate good generalization
- **Balanced predictions**: High precision and recall for both classes

### Feature Engineering Impact
- The comprehensive preprocessing pipeline directly contributed to high model performance
- Statistical difference features capture the competitive nature of Pokemon battles
- Type encoding and battle mechanics features provide domain-specific insights

### Business Value
- **High reliability**: 95%+ accuracy enables confident battle outcome predictions
- **Practical applications**: Could be used for team composition, strategy planning, or game balance analysis
- **Scalable approach**: The methodology can be applied to other competitive game scenarios

---

## Conclusion

This project successfully demonstrates end-to-end machine learning workflow for Pokemon battle prediction:

1. **Data Understanding**: Analyzed Pokemon battle mechanics and statistical relationships
2. **Feature Engineering**: Created 48 meaningful features from raw data
3. **Model Development**: Compared multiple algorithms and optimized the best performer
4. **Evaluation**: Achieved 95%+ accuracy with robust validation methodology

The combination of domain knowledge, thoughtful feature engineering, and proper model evaluation resulted in a highly accurate and reliable prediction system.